# Hyperparameter tuning using k-best LightGBM features

## Import the required modules/libraries

In [2]:
%pip install seaborn
%pip install -U imblearn
%pip install shap

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 18.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
from Supervised_ML_Pipeline import MLPipeline
import pandas as pd
import numpy as np

# Standerdizing 
from sklearn.preprocessing import StandardScaler

## Check available options

In [2]:
MLPipeline.list_feature_selection_methods()

['k_best_catboost', 'k_best_xgb', 'k_best_lgbm', 'k_best_decision_tree']

In [3]:
MLPipeline.list_undersample_methods()

['RandomUnderSampler', 'CLEANSE']

In [4]:
MLPipeline.list_models()

['logistic_regression', 'xgboost', 'svm']

All models trained within this notebook will make use of the `k_best_lgbm` feature selection method. For each model we fine-tune hyperparameters using `RandomUnderSampler`, `CLEANSE`, or no undersampling. However, for `svm` we must undersample due to the runtime complexity of the algorithm. Hence, 8 undersampling and model combinations will be tried.

## Import the data

In [5]:
df = pd.read_csv("/cluster/pixstor/haithcoatt-lab/SP26cCapStone_Team2/Data/Silver/unified_dme_year_npi_hcpcs_rentl_ind_labeled.csv")

target_col = 'target'

exclude_cols = [
    'npi', 'hcpcs_cd', 'rfrg_prvdr_state_abrvtn',
    'specialty_type', 'specialty'
]

df.head(5)

,npi,hcpcs_cd,suplr_rentl_ind,tot_suplrs,tot_suplr_benes,tot_suplr_clms,tot_suplr_srvcs,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,year,...,claims_per_bene_zscore_by_type,claims_per_bene_outlier_by_type,services_per_bene_zscore,services_per_bene_outlier,services_per_bene_zscore_by_type,services_per_bene_outlier_by_type,benes_per_supplier_zscore,benes_per_supplier_outlier,benes_per_supplier_zscore_by_type,benes_per_supplier_outlier_by_type
0,1003000902,A4253,0,11,14.0,32,67,68.483134,8.386567,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
1,1003000902,A4259,0,8,5.0,14,17,14.727647,1.437647,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
2,1003000902,A4390,0,3,5.0,17,330,16.484364,11.178182,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
3,1003000902,E0570,1,1,5.0,12,12,50.730000,5.921667,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
4,1003000902,E1390,1,2,5.0,24,24,389.755000,111.531667,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False


## Feature extraction
This functions drops the target and other excluded features, returning only numerical features that the model can work with.

In [6]:
# These type hints are added for readability sake, this way the people understand what this function would require.
# Additionaly, it would give warnings or outright fail when used incorrectly. This makes the debugging process a bit easier. 

def extract_features(df: pd.DataFrame,target_col: str,exclude_cols: list[str]) -> pd.DataFrame:
    cols_to_drop = [target_col] + [c for c in exclude_cols if c in df.columns]
    temp = df.drop(columns=cols_to_drop).select_dtypes(include=[np.number])
    return temp

## XGBoost with RandomUnderSampler

In [7]:
pipe_xgb = MLPipeline(model_name="xgboost", target_col="target")

In [8]:
param_grid = {
        'model__min_child_weight': [2, 3, 4, 5],
        'model__gamma': [15, 20, 25, 30],
        'model__subsample': [0.85, 0.90, 0.95, 1.0],
        'model__colsample_bytree': [0.6],
        'model__max_depth': [4],
        'selector__k': [100]
        }

In [9]:
pipe_xgb.train(df, param_grid=param_grid, group_col="npi", exclude_cols=exclude_cols, undersample_method='RandomUnderSampler', feature_selection_method='k_best_lgbm')


Undersampling (RandomUnderSampler) applied.
Class distribution after undersampling: {np.int64(0): np.int64(3622), np.int64(1): np.int64(326)}

Feature Selection (k_best_lgbm) selected.
Fitting 5 folds for each of 64 candidates, totalling 320 fits

------------------------------------------------------------------------------------------------------------------------------------------------------


Model: xgboost
CV folds: 5  |  Scoring: average_precision
Best CV score: 0.2385 ± 0.1492 (std across folds)
Score range: min = 0.0816  max = 0.4836  cv = 0.6253 (std/mean)
Training time: 65.3 Seconds
Best params: {'model__colsample_bytree': 0.6, 'model__gamma': 20, 'model__max_depth': 4, 'model__min_child_weight': 3, 'model__subsample': 0.9, 'selector__k': 100}

Per-fold breakdown:
  Fold          Score
  1            0.4836
  2            0.3060
  3            0.0816
  4            0.2311
  5            0.0904


-------------------------------------------------------------------------------

In [10]:
pipe_xgb.test()


Precision@K plot saved in path:./xgboost_precision_at_k_20260423_105613.png


------------------------------------------------------------------------------------------------------------------------------------------------------



Metrics saved in path:./xgboost_metrics_20260423_105613.txt


------------------------------------------------------------------------------------------------------------------------------------------------------


Test-set metrics:
 auc_roc              0.8818
 auprc                0.0331
 f1_macro             0.528
 precision_macro      0.5161
 recall_macro         0.6694
 best_cv_score        0.2385
 training_time        65.3
 precision@10         0.2
 precision@20         0.2
 precision@50         0.12
 precision@100        0.1
 best_params          {'model__colsample_bytree': 0.6, 'model__gamma': 20, 'model__max_depth': 4, 'model__min_child_weight': 3, 'model__subsample': 0.9, 'selector__k': 100}


------------------------------------------------------

 98%|===================| 319916/326999 [00:24<00:00]        

SHAP importance plot saved in path:./xgboost_shap_importance_20260423_105639.png


------------------------------------------------------------------------------------------------------------------------------------------------------


